# Sprint 10: Integrated Mature System Scaling & Holdout Generalization Study
### SIH Problem Statement 26074: Block to Panchayat Weather Downscaling

This notebook executes the complete Sprint 10 research agenda:
1. **Dense-L Convergence & Extended Fine-Tuning**: Resumes from Sprint 9 Dense-L weights (52.0M params, epoch 30, loss ~0.41065) with validation-loss tracking and early stopping (patience=5) to reach convergence without overfitting.
2. **Canonical Physical Evaluation (sprint8_physical)**: Re-evaluates on authentic 2022 validation split under non-linear inverse normalization and member-wise physical bounds repair (wet threshold p_tgt > 2.5 mm/day).
3. **2023 Quarantined El Nino Holdout Generalization**: Unquarantines the 2023 test set to measure out-of-distribution climate resilience.
4. **Multivariate Physical Diagnostics**: Thermodynamic bounds (Tmax >= Tmin), moisture coupling r(P, RH), and Brier Skill Scores.


In [ ]:
# Cell 1: Environment Hardware Diagnostics and Package Setup
import os
import sys
import time
import math
import hashlib
import json
import zipfile
import shutil
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import yaml

print("=" * 60)
print("[*] Environment Hardware Diagnostics:")
print(f"    PyTorch:  {torch.__version__}")
print(f"    CUDA:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"    GPU:      {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"    VRAM:     {vram_gb:.2f} GB")
print("=" * 60)

for pkg in ["zarr"]:
    try:
        __import__(pkg)
    except ImportError:
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import zarr

def set_global_seed(seed: int = 42):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

set_global_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[+] Execution Device: {device}")


In [ ]:
# Cell 2: Dataset Discovery and Sprint 9 Dense-L Checkpoint Staging
working_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
os.chdir(str(working_dir))

datasets_dir = working_dir / "datasets"
datasets_dir.mkdir(parents=True, exist_ok=True)
data_dir = working_dir / "data"
data_dir.mkdir(parents=True, exist_ok=True)
models_dir = working_dir / "models" / "checkpoints"
models_dir.mkdir(parents=True, exist_ok=True)
reports_dir = working_dir / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

# 1. Locate Authentic Zarr archive
zarr_file = None
for p in [datasets_dir / "multitask_temporal_v2_h14.zarr", Path("datasets/multitask_temporal_v2_h14.zarr")]:
    if p.exists() and ((p / ".zgroup").exists() or (p / "dates").exists() or (p / "zarr.json").exists()):
        zarr_file = p
        break

if zarr_file is None and Path("/kaggle/input").exists():
    for d in Path("/kaggle/input").rglob("*.zarr"):
        if (d / ".zgroup").exists() or (d / "dates").exists() or (d / "zarr.json").exists():
            zarr_file = d
            break

if zarr_file is None or not zarr_file.exists():
    raise FileNotFoundError("Authentic Zarr dataset not found! Attach 'rohitajitbharadwaj/sih26074-multitask-temporal-v2-h14'.")
print(f"[+] Located authentic Zarr dataset: {zarr_file}")

# 2. Locate Parquet index and YAML normalization stats
index_file = None
for p in [data_dir / "sample_index_v2_h14.parquet", Path("data/sample_index_v2_h14.parquet")]:
    if p.exists():
        index_file = p
        break
if index_file is None and Path("/kaggle/input").exists():
    for f in Path("/kaggle/input").rglob("*sample_index*.parquet"):
        index_file = f
        break

stats_file = None
for p in [data_dir / "normalization_stats_v2.yaml", Path("data/normalization_stats_v2.yaml")]:
    if p.exists():
        stats_file = p
        break
if stats_file is None and Path("/kaggle/input").exists():
    for f in Path("/kaggle/input").rglob("*normalization_stats*.yaml"):
        stats_file = f
        break

if index_file is None or stats_file is None:
    raise FileNotFoundError("Sample index or normalization stats not found!")

print(f"[+] Located sample index: {index_file}")
print(f"[+] Located normalization stats: {stats_file}")

# 3. Locate Sprint 9 Dense-L Checkpoint
ckpt_dense_l = None
for p in [models_dir / "sprint9_dense_l_weights.pt", Path("models/checkpoints/sprint9_dense_l_weights.pt")]:
    if p.exists():
        ckpt_dense_l = p
        break

if ckpt_dense_l is None and Path("/kaggle/input").exists():
    for f in Path("/kaggle/input").rglob("*sprint9_dense_l*.pt"):
        ckpt_dense_l = f
        break

if ckpt_dense_l is None or not ckpt_dense_l.exists():
    raise FileNotFoundError("Sprint 9 Dense-L checkpoint not found! Attach 'ssachithananthan/sih26074-sprint9-checkpoints'.")

raw_b = ckpt_dense_l.read_bytes()
if raw_b.startswith(b"version https://git-lfs.github.com"):
    raise RuntimeError("Dense-L checkpoint is an unhydrated Git-LFS pointer!")

print(f"[+] Staged Sprint 9 Dense-L checkpoint: {ckpt_dense_l.name} ({len(raw_b):,} bytes)")


In [ ]:
# Cell 3: SpatiotemporalDownscalingDataset Implementation
class SpatiotemporalDownscalingDataset(Dataset):
    def __init__(self, zarr_path: Path, index_path: Path, stats_path: Path, split: str = "val", history_len: int = 14, context_size: int = 24):
        self.zarr_path = Path(zarr_path)
        self.index_path = Path(index_path)
        self.stats_path = Path(stats_path)
        self.split = str(split).lower()
        self.history_len = history_len
        self.context_size = context_size

        self.store = zarr.open_group(str(self.zarr_path), mode="r")
        self.df_all = pd.read_parquet(self.index_path)
        if self.split != "all":
            self.df = self.df_all[self.df_all["split"] == self.split].reset_index(drop=True)
        else:
            self.df = self.df_all.copy().reset_index(drop=True)

        all_ids = list(self.df_all["sample_id"])
        id_to_global = {sid: i for i, sid in enumerate(all_ids)}
        self.global_indices = [id_to_global[sid] for sid in self.df["sample_id"]]

        self.terrain_tensor = torch.from_numpy(np.asarray(self.store["terrain"][:], dtype=np.float32))
        with open(self.stats_path, "r", encoding="utf-8") as f:
            self.stats_doc = yaml.safe_load(f)
        self.stats = self.stats_doc.get("channels", {})
        p_stat = self.stats.get("precipitation", self.stats.get("precip", {}))
        self.stats["precipitation"] = p_stat
        self.stats["precip"] = p_stat

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        g_idx = self.global_indices[idx]
        hist_raw = np.asarray(self.store["history"][g_idx], dtype=np.float32)
        fcst_raw = np.asarray(self.store["future_forecast"][g_idx], dtype=np.float32)
        target_raw = np.asarray(self.store["target"][g_idx], dtype=np.float32)

        if hist_raw.shape[0] > self.history_len:
            hist_raw = hist_raw[-self.history_len:]

        ch_names = ["precip", "tmax", "tmin", "rh", "wind_u", "wind_v"]
        for c_idx, name in enumerate(ch_names):
            if name in self.stats:
                m = self.stats[name]["mean"]
                s = max(1e-6, self.stats[name]["std"])
                hist_raw[:, c_idx] = (hist_raw[:, c_idx] - m) / s
                fcst_raw[:, c_idx] = (fcst_raw[:, c_idx] - m) / s
                target_raw[:, c_idx] = (target_raw[:, c_idx] - m) / s

        return {
            "history": torch.from_numpy(hist_raw),
            "future_forecast": torch.from_numpy(fcst_raw),
            "terrain": self.terrain_tensor,
            "target": torch.from_numpy(target_raw),
        }

print("[+] Dataset streaming engine initialized.")


In [ ]:
# Cell 4: Dense-L Champion Architecture (51,997,958 Parameters)
def compute_residual_target(y_fine_norm: torch.Tensor, future_forecast_norm: torch.Tensor) -> torch.Tensor:
    b, leads, c, h, w = future_forecast_norm.shape
    if h > 16 or w > 16:
        offset_h = (h - 16) // 2
        offset_w = (w - 16) // 2
        fcst_crop = future_forecast_norm[:, :, :, offset_h : offset_h + 16, offset_w : offset_w + 16]
    else:
        fcst_crop = future_forecast_norm

    fcst_flat = fcst_crop.reshape(b * leads, c, 16, 16)
    fcst_up = F.interpolate(fcst_flat, size=(80, 80), mode="bilinear", align_corners=False).view(b, leads, c, 80, 80)
    return y_fine_norm - fcst_up


class SinusoidalPositionalEmbedding(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, timesteps: torch.Tensor) -> torch.Tensor:
        half_dim = self.dim // 2
        exponent = -math.log(10000) * torch.arange(half_dim, dtype=torch.float32, device=timesteps.device) / half_dim
        emb = torch.exp(exponent)
        emb = timesteps.float().unsqueeze(1) * emb.unsqueeze(0)
        return torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)


class TimeConditionedConvNeXtBlock(nn.Module):
    def __init__(self, dim: int, time_emb_dim: int, expansion: int = 2, num_groups: int = 8):
        super().__init__()
        assert dim % num_groups == 0
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim, bias=False)
        self.norm = nn.GroupNorm(num_groups=num_groups, num_channels=dim)
        self.time_proj = nn.Sequential(nn.GELU(), nn.Linear(time_emb_dim, dim * 2))
        self.pwconv1 = nn.Conv2d(dim, dim * expansion, kernel_size=1)
        self.act = nn.GELU()
        self.pwconv2 = nn.Conv2d(dim * expansion, dim, kernel_size=1)

    def forward(self, x: torch.Tensor, time_emb: torch.Tensor) -> torch.Tensor:
        res = x
        out = self.dwconv(x)
        out = self.norm(out)
        scale_shift = self.time_proj(time_emb)
        scale, shift = scale_shift.chunk(2, dim=-1)
        scale = torch.clamp(scale, min=-4.0, max=4.0)
        shift = torch.clamp(shift, min=-8.0, max=8.0)
        out = out * (1.0 + scale.unsqueeze(-1).unsqueeze(-1)) + shift.unsqueeze(-1).unsqueeze(-1)
        out = self.pwconv1(out)
        out = self.act(out)
        out = self.pwconv2(out)
        return res + out


class ScalableSpatiotemporalDenoiser(nn.Module):
    def __init__(self, in_weather_channels: int = 6, terrain_channels: int = 5, num_leads: int = 7, base_channels: int = 176, time_emb_dim: int = 64, embed_dim: int = 32, num_heads: int = 4):
        super().__init__()
        self.in_weather_channels = in_weather_channels
        self.num_leads = num_leads
        self.base_channels = base_channels
        self.time_emb_dim = time_emb_dim
        self.embed_dim = embed_dim

        self.time_mlp = nn.Sequential(
            SinusoidalPositionalEmbedding(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.GELU(),
            nn.Linear(time_emb_dim, time_emb_dim),
        )
        self.lead_embed = nn.Embedding(num_leads, embed_dim)

        self.hist_proj = nn.Sequential(
            nn.Conv2d(in_weather_channels, embed_dim, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(embed_dim, embed_dim, kernel_size=1),
        )
        self.fcst_proj = nn.Sequential(
            nn.Conv2d(in_weather_channels, embed_dim, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(embed_dim, embed_dim, kernel_size=1),
        )
        self.hist_self_attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
        self.hist_future_cross_attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
        self.cross_norm = nn.LayerNorm(embed_dim)

        self.spatial_up = nn.Sequential(
            nn.Upsample(size=(80, 80), mode="bilinear", align_corners=False),
            nn.Conv2d(embed_dim, base_channels, kernel_size=3, padding=1),
            nn.GELU(),
        )
        self.terrain_enc = nn.Sequential(
            nn.Conv2d(terrain_channels, base_channels, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(base_channels, base_channels, kernel_size=3, padding=1),
        )

        in_stem_dim = in_weather_channels + base_channels + base_channels
        self.stem = nn.Sequential(
            nn.Conv2d(in_stem_dim, base_channels, kernel_size=3, padding=1),
            nn.GELU(),
        )
        self.stem_block = TimeConditionedConvNeXtBlock(base_channels, time_emb_dim, num_groups=4)

        c2 = base_channels * 2
        self.down1_conv = nn.Conv2d(base_channels, c2, kernel_size=3, stride=2, padding=1)
        self.down1_block = TimeConditionedConvNeXtBlock(c2, time_emb_dim, num_groups=8)

        c3 = base_channels * 4
        self.down2_conv = nn.Conv2d(c2, c3, kernel_size=3, stride=2, padding=1)
        self.down2_block = TimeConditionedConvNeXtBlock(c3, time_emb_dim, num_groups=8)

        c4 = base_channels * 8
        self.down3_conv = nn.Conv2d(c3, c4, kernel_size=3, stride=2, padding=1)
        self.down3_block = TimeConditionedConvNeXtBlock(c4, time_emb_dim, num_groups=8)

        self.bottleneck_temporal_attn = nn.MultiheadAttention(embed_dim=c4, num_heads=num_heads, batch_first=True)
        self.bottleneck_norm = nn.LayerNorm(c4)

        self.up3 = nn.Upsample(size=(20, 20), mode="bilinear", align_corners=False)
        self.dec3_conv = nn.Conv2d(c4 + c3, c3, kernel_size=3, padding=1)
        self.dec3_block = TimeConditionedConvNeXtBlock(c3, time_emb_dim, num_groups=8)

        self.up2 = nn.Upsample(size=(40, 40), mode="bilinear", align_corners=False)
        self.dec2_conv = nn.Conv2d(c3 + c2, c2, kernel_size=3, padding=1)
        self.dec2_block = TimeConditionedConvNeXtBlock(c2, time_emb_dim, num_groups=8)

        self.up1 = nn.Upsample(size=(80, 80), mode="bilinear", align_corners=False)
        self.dec1_conv = nn.Conv2d(c2 + base_channels, base_channels, kernel_size=3, padding=1)
        self.dec1_block = TimeConditionedConvNeXtBlock(base_channels, time_emb_dim, num_groups=4)

        self.head_precip = nn.Conv2d(base_channels, 1, kernel_size=3, padding=1)
        self.head_tmax = nn.Conv2d(base_channels, 1, kernel_size=3, padding=1)
        self.head_tmin = nn.Conv2d(base_channels, 1, kernel_size=3, padding=1)
        self.head_rh = nn.Conv2d(base_channels, 1, kernel_size=3, padding=1)
        self.head_wind_u = nn.Conv2d(base_channels, 1, kernel_size=3, padding=1)
        self.head_wind_v = nn.Conv2d(base_channels, 1, kernel_size=3, padding=1)

    def forward(self, r_t: torch.Tensor, t: torch.Tensor, history: torch.Tensor, future_forecast: torch.Tensor, terrain: torch.Tensor):
        b, num_leads, _, _, _ = r_t.shape
        h_len = history.shape[1]

        t_emb = self.time_mlp(t)
        t_emb_flat = t_emb.unsqueeze(1).expand(b, num_leads, self.time_emb_dim).reshape(b * num_leads, self.time_emb_dim)

        _, _, _, n_lat, n_lon = history.shape
        hist_flat = history.reshape(b * h_len, self.in_weather_channels, n_lat, n_lon)
        hist_feats = self.hist_proj(hist_flat)
        hist_tokens = F.adaptive_avg_pool2d(hist_feats, 1).view(b, h_len, self.embed_dim)
        hist_context, _ = self.hist_self_attn(hist_tokens, hist_tokens, hist_tokens)

        coarse_flat = future_forecast.reshape(b * num_leads, self.in_weather_channels, n_lat, n_lon)
        fcst_feats = self.fcst_proj(coarse_flat)
        fcst_tokens = F.adaptive_avg_pool2d(fcst_feats, 1).view(b, num_leads, self.embed_dim)
        lead_idx = torch.arange(num_leads, device=r_t.device)
        lead_emb = self.lead_embed(lead_idx).unsqueeze(0).expand(b, num_leads, self.embed_dim)
        fcst_tokens = fcst_tokens + lead_emb

        cross_context, _ = self.hist_future_cross_attn(query=fcst_tokens, key=hist_context, value=hist_context)
        fused_tokens = self.cross_norm(fcst_tokens + cross_context)

        fused_spatial = fcst_feats + fused_tokens.view(b * num_leads, self.embed_dim, 1, 1)
        if n_lat > 16 or n_lon > 16:
            crop_lat = (n_lat - 16) // 2
            crop_lon = (n_lon - 16) // 2
            fused_spatial_16 = fused_spatial[:, :, crop_lat : crop_lat + 16, crop_lon : crop_lon + 16]
        else:
            fused_spatial_16 = fused_spatial

        atmos_80 = self.spatial_up(fused_spatial_16)
        terr_feats = self.terrain_enc(terrain)
        terr_flat = terr_feats.unsqueeze(1).expand(b, num_leads, self.base_channels, 80, 80).reshape(b * num_leads, self.base_channels, 80, 80)

        r_t_flat = r_t.view(b * num_leads, self.in_weather_channels, 80, 80)
        stem_in = torch.cat([r_t_flat, atmos_80, terr_flat], dim=1)

        s1 = self.stem(stem_in)
        s1 = self.stem_block(s1, t_emb_flat)
        s2 = self.down1_conv(s1)
        s2 = self.down1_block(s2, t_emb_flat)
        s3 = self.down2_conv(s2)
        s3 = self.down2_block(s3, t_emb_flat)
        s4 = self.down3_conv(s3)
        s4 = self.down3_block(s4, t_emb_flat)

        _, c4, h10, w10 = s4.shape
        s4_perm = s4.view(b, num_leads, c4, h10 * w10).permute(0, 3, 1, 2).reshape(b * h10 * w10, num_leads, c4)
        s4_temporal, _ = self.bottleneck_temporal_attn(s4_perm, s4_perm, s4_perm)
        s4_temporal = self.bottleneck_norm(s4_perm + s4_temporal)
        s4_fused = s4_temporal.view(b, h10 * w10, num_leads, c4).permute(0, 2, 3, 1).reshape(b * num_leads, c4, h10, w10)

        u3 = self.up3(s4_fused)
        d3 = self.dec3_block(self.dec3_conv(torch.cat([u3, s3], dim=1)), t_emb_flat)
        u2 = self.up2(d3)
        d2 = self.dec2_block(self.dec2_conv(torch.cat([u2, s2], dim=1)), t_emb_flat)
        u1 = self.up1(d2)
        d1 = self.dec1_block(self.dec1_conv(torch.cat([u1, s1], dim=1)), t_emb_flat)

        p = self.head_precip(d1)
        tmax = self.head_tmax(d1)
        tmin = self.head_tmin(d1)
        rh = self.head_rh(d1)
        wu = self.head_wind_u(d1)
        wv = self.head_wind_v(d1)

        out = torch.cat([p, tmax, tmin, rh, wu, wv], dim=1)
        return out.view(b, num_leads, 6, 80, 80)


class ScalableSpatiotemporalResidualDiffusion(nn.Module):
    def __init__(self, base_channels: int = 176, timesteps: int = 100):
        super().__init__()
        self.timesteps = timesteps
        self.denoiser = ScalableSpatiotemporalDenoiser(base_channels=base_channels)

        betas = torch.linspace(1e-4, 0.035, timesteps, dtype=torch.float32)
        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)

        self.register_buffer("betas", betas)
        self.register_buffer("alphas", alphas)
        self.register_buffer("alphas_cumprod", alphas_cumprod)
        self.register_buffer("sqrt_alphas_cumprod", torch.sqrt(alphas_cumprod))
        self.register_buffer("sqrt_one_minus_alphas_cumprod", torch.sqrt(1.0 - alphas_cumprod))

    def compute_v_target(self, r_0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor) -> torch.Tensor:
        sqrt_alpha = self.sqrt_alphas_cumprod[t].view(-1, 1, 1, 1, 1)
        sqrt_one_minus = self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1, 1)
        return sqrt_alpha * noise - sqrt_one_minus * r_0

    def compute_training_loss(self, r_0: torch.Tensor, history: torch.Tensor, future_forecast: torch.Tensor, terrain: torch.Tensor, target_norm: Optional[torch.Tensor] = None):
        b = r_0.shape[0]
        device = r_0.device
        t = torch.randint(0, self.timesteps, (b,), device=device, dtype=torch.long)
        noise = torch.randn_like(r_0)
        sqrt_alpha = self.sqrt_alphas_cumprod[t].view(-1, 1, 1, 1, 1)
        sqrt_one_minus = self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1, 1)
        r_t = sqrt_alpha * r_0 + sqrt_one_minus * noise

        model_pred = self.denoiser(r_t=r_t, t=t, history=history, future_forecast=future_forecast, terrain=terrain)
        v_target = self.compute_v_target(r_0, t, noise)

        pred_f = model_pred.float()
        target_f = v_target.float()

        ref_precip = target_norm[:, :, 0:1] if target_norm is not None else r_0[:, :, 0:1]
        tail_mask = (ref_precip.float() > 0.469).float()
        precip_weights = 1.0 + 2.0 * tail_mask
        p_loss = (precip_weights * F.smooth_l1_loss(pred_f[:, :, 0:1], target_f[:, :, 0:1], beta=1.0, reduction="none")).mean()
        thermo_loss = F.smooth_l1_loss(pred_f[:, :, 1:4], target_f[:, :, 1:4], beta=1.0)
        wind_loss = F.smooth_l1_loss(pred_f[:, :, 4:6], target_f[:, :, 4:6], beta=1.0)

        loss = 1.0 * p_loss + 1.2 * thermo_loss + 1.1 * wind_loss
        return loss, model_pred, v_target

    @torch.no_grad()
    def sample(self, history: torch.Tensor, future_forecast: torch.Tensor, terrain: torch.Tensor, num_steps: int = 4, eta: float = 0.0, seed: Optional[int] = None):
        if seed is not None:
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed(seed)

        b, leads, c, h, w = future_forecast.shape
        device = history.device
        r_t = torch.randn(b, leads, 6, 80, 80, device=device)

        step_indices = np.linspace(self.timesteps - 1, 0, num_steps, dtype=int)
        for i in range(len(step_indices)):
            cur_step = int(step_indices[i])
            t = torch.full((b,), cur_step, device=device, dtype=torch.long)
            v_pred = self.denoiser(r_t=r_t, t=t, history=history, future_forecast=future_forecast, terrain=terrain)

            alpha_bar = self.alphas_cumprod[cur_step]
            sqrt_alpha_bar = self.sqrt_alphas_cumprod[cur_step]
            sqrt_one_minus = self.sqrt_one_minus_alphas_cumprod[cur_step]

            pred_r0 = sqrt_alpha_bar * r_t - sqrt_one_minus * v_pred
            pred_eps = sqrt_alpha_bar * v_pred + sqrt_one_minus * r_t

            if i < len(step_indices) - 1:
                next_step = int(step_indices[i + 1])
                alpha_bar_next = self.alphas_cumprod[next_step]
                sqrt_alpha_bar_next = self.sqrt_alphas_cumprod[next_step]
                sqrt_one_minus_next = self.sqrt_one_minus_alphas_cumprod[next_step]

                sigma = eta * torch.sqrt((1 - alpha_bar_next) / (1 - alpha_bar) * (1 - alpha_bar / alpha_bar_next))
                noise = torch.randn_like(r_t) if eta > 0.0 else torch.zeros_like(r_t)
                dir_xt = torch.sqrt(torch.clamp(1.0 - alpha_bar_next - sigma**2, min=0.0)) * pred_eps
                r_t = sqrt_alpha_bar_next * pred_r0 + dir_xt + sigma * noise
            else:
                r_t = pred_r0

        if h > 16 or w > 16:
            off_h = (h - 16) // 2
            off_w = (w - 16) // 2
            fcst_crop = future_forecast[:, :, :, off_h : off_h + 16, off_w : off_w + 16]
        else:
            fcst_crop = future_forecast

        fcst_up = F.interpolate(fcst_crop.reshape(b * leads, c, 16, 16), size=(80, 80), mode="bilinear", align_corners=False).view(b, leads, c, 80, 80)
        return fcst_up + r_t

print("[+] Scalable Dense-L compiled: 51,997,958 parameters.")


In [ ]:
# Cell 5: Dense-L Extended Convergence Fine-Tuning
model_l = ScalableSpatiotemporalResidualDiffusion(base_channels=176).to(device)

# Load Sprint 9 weights
print(f"[*] Loading Sprint 9 Dense-L weights from: {ckpt_dense_l.name}")
state = torch.load(ckpt_dense_l, map_location=device)
weights = state.get("model_state_dict", state)
missing, unexpected = model_l.load_state_dict(weights, strict=False)
print(f"[+] Loaded Dense-L state dict (Missing: {len(missing)}, Unexpected: {len(unexpected)})")
assert len(missing) == 0, f"Critical model parameters missing: {missing}"

start_epoch = int(state.get("epochs", state.get("epoch", 30)))
init_loss = float(state.get("loss", state.get("train_loss", 0.41065)))
print(f"[+] Loaded Dense-L from Epoch {start_epoch} (Prior Loss: {init_loss:.5f})")

ds_train = SpatiotemporalDownscalingDataset(zarr_file, index_file, stats_file, split="train")
train_loader = DataLoader(ds_train, batch_size=2, shuffle=True)

ds_val = SpatiotemporalDownscalingDataset(zarr_file, index_file, stats_file, split="val")
val_loader = DataLoader(ds_val, batch_size=2, shuffle=False)
print(f"[+] Loaded {len(ds_train)} train cubes and {len(ds_val)} val cubes.")

def eval_val_loss(m: nn.Module, loader: DataLoader) -> float:
    m.eval()
    tot, cnt = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            h = batch["history"].to(device)
            f = batch["future_forecast"].to(device)
            terr = batch["terrain"].to(device)
            y = batch["target"].to(device)
            r0 = compute_residual_target(y, f)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                loss, _, _ = m.compute_training_loss(r0, h, f, terr, target_norm=y)
            if torch.isfinite(loss):
                tot += loss.item()
                cnt += 1
    return tot / max(1, cnt)

baseline_val_loss = eval_val_loss(model_l, val_loader)
print(f"[+] Baseline Validation Loss at Epoch {start_epoch}: {baseline_val_loss:.5f}")

# Optimization settings
FINE_TUNE_EPOCHS = 15  # Dynamic extended epochs
PATIENCE = 5           # Early stopping patience
LR = 3e-5              # Gentle fine-tuning LR

optimizer = torch.optim.AdamW(model_l.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FINE_TUNE_EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available(), init_scale=2048.0)

best_val = baseline_val_loss
patience_count = 0
champion_ckpt_file = models_dir / "sprint10_dense_l_champion.pt"

training_records = []
total_epochs = start_epoch + FINE_TUNE_EPOCHS

for epoch in range(start_epoch + 1, total_epochs + 1):
    model_l.train()
    total_loss, n_b = 0.0, 0
    t0 = time.time()

    for batch in train_loader:
        optimizer.zero_grad()
        h = batch["history"].to(device)
        f = batch["future_forecast"].to(device)
        terr = batch["terrain"].to(device)
        y = batch["target"].to(device)
        r0 = compute_residual_target(y, f)

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            loss, _, _ = model_l.compute_training_loss(r0, h, f, terr, target_norm=y)

        if not torch.isfinite(loss):
            optimizer.zero_grad(set_to_none=True)
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model_l.parameters(), max_norm=1.0)
        if not torch.isfinite(grad_norm):
            optimizer.zero_grad(set_to_none=True)
            scaler.update()
            continue

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        n_b += 1

    scheduler.step()
    train_loss = total_loss / max(1, n_b)
    val_loss = eval_val_loss(model_l, val_loader)
    elapsed = time.time() - t0
    cur_lr = optimizer.param_groups[0]["lr"]

    print(f"[Epoch {epoch:02d}/{total_epochs:02d}] Train Loss: {train_loss:.5f} | Val Loss: {val_loss:.5f} | LR: {cur_lr:.6f} | Time: {elapsed:.1f}s")
    training_records.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "lr": cur_lr})

    if val_loss < best_val - 1e-4:
        best_val = val_loss
        patience_count = 0
        ckpt_payload = {
            "epoch": epoch,
            "tier": "dense_l",
            "model_state_dict": model_l.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss,
            "param_count": 51_997_958,
        }
        torch.save(ckpt_payload, champion_ckpt_file)
        torch.save(ckpt_payload, working_dir / "sprint10_dense_l_champion.pt")
        print(f"    [+] Saved NEW CHAMPION checkpoint! (Val Loss: {val_loss:.5f})")
    else:
        patience_count += 1
        print(f"    [-] Patience {patience_count}/{PATIENCE}")
        if patience_count >= PATIENCE:
            print(f"[!] Early stopping triggered at Epoch {epoch}! Convergence reached without overfitting.")
            break

# Export training history
with open(reports_dir / "sprint10_training_history.json", "w") as f:
    json.dump(training_records, f, indent=2)
with open(working_dir / "sprint10_training_history.json", "w") as f:
    json.dump(training_records, f, indent=2)


In [ ]:
# Cell 6: Canonical Physical Evaluation Engine (sprint8_physical Protocol)
# Load champion weights
if champion_ckpt_file.exists():
    state_champ = torch.load(champion_ckpt_file, map_location=device)
    model_l.load_state_dict(state_champ["model_state_dict"], strict=False)
    print(f"[+] Loaded champion model from Epoch {state_champ.get('epoch')}")

def compute_laplacian_energy(img: torch.Tensor) -> float:
    kernel = torch.tensor([[0.0, 1.0, 0.0], [1.0, -4.0, 1.0], [0.0, 1.0, 0.0]], device=img.device).view(1, 1, 3, 3)
    x = img.view(1, 1, img.shape[-2], img.shape[-1]).float()
    lap = F.conv2d(x, kernel, padding=1)
    return float(torch.mean(lap ** 2).item())

def evaluate_canonical_physical(m: nn.Module, loader: DataLoader, stats: Dict[str, Any]):
    m.eval()
    wet_maes, csi15s, csi30s, csi50s = [], [], [], []
    crps_list, spread_list, rmse_list = [], [], []
    lap_ratios = []
    tmax_maes, rh_maes = [], []
    thermo_violations, total_pts = 0, 0
    p_rh_corrs, mass_shifts = [], []

    p_stat = stats.get("precipitation", stats.get("precip", {}))
    p_m = float(p_stat.get("mean", 6.26608))
    p_s = float(p_stat.get("std", 18.6239))

    with torch.no_grad():
        for b_idx, batch in enumerate(loader):
            h = batch["history"].to(device)
            f = batch["future_forecast"].to(device)
            terr = batch["terrain"].to(device)
            y = batch["target"].to(device)
            b = y.shape[0]

            # 1. Point Mode: K=8, S=4, eta=0.5 (32 NFE)
            pt_mems = []
            for k in range(8):
                pt_mems.append(m.sample(h, f, terr, num_steps=4, eta=0.5, seed=1000 + b_idx * 10 + k))
            pt_stack = torch.stack(pt_mems, dim=0)

            # Invert normalization
            phys_list = []
            for k in range(8):
                arr = pt_stack[k].clone()
                # Exponential rain inversion
                arr[:, :, 0] = torch.clamp(torch.exp(arr[:, :, 0] * p_s + p_m) - 1.0, min=0.0)
                # Linear thermo inversion
                for c_i, name in [(1, "tmax"), (2, "tmin"), (3, "rh"), (4, "wind_u"), (5, "wind_v")]:
                    arr[:, :, c_i] = arr[:, :, c_i] * stats[name]["std"] + stats[name]["mean"]
                phys_list.append(arr)
            phys_stack = torch.stack(phys_list, dim=0)

            # Physical repair: clamp RH to [0, 100], enforce Tmax >= Tmin
            phys_stack[:, :, :, 3] = torch.clamp(phys_stack[:, :, :, 3], min=0.0, max=100.0)
            phys_stack[:, :, :, 1] = torch.maximum(phys_stack[:, :, :, 1], phys_stack[:, :, :, 2])

            ens_mean = phys_stack.mean(dim=0)

            # Target inversion
            y_phys = y.clone()
            y_phys[:, :, 0] = torch.clamp(torch.exp(y_phys[:, :, 0] * p_s + p_m) - 1.0, min=0.0)
            for c_i, name in [(1, "tmax"), (2, "tmin"), (3, "rh"), (4, "wind_u"), (5, "wind_v")]:
                y_phys[:, :, c_i] = y_phys[:, :, c_i] * stats[name]["std"] + stats[name]["mean"]

            p_pred = ens_mean[:, :, 0]
            p_tgt = y_phys[:, :, 0]

            w_mask = p_tgt > 2.5
            if w_mask.any():
                wet_maes.append(float(torch.mean(torch.abs(p_pred[w_mask] - p_tgt[w_mask])).item()))
            else:
                wet_maes.append(float(torch.mean(torch.abs(p_pred - p_tgt)).item()))

            for thr, arr in [(15.0, csi15s), (30.0, csi30s), (50.0, csi50s)]:
                h_c = float(torch.logical_and(p_pred >= thr, p_tgt >= thr).sum().item())
                fa_c = float(torch.logical_and(p_pred >= thr, p_tgt < thr).sum().item())
                m_c = float(torch.logical_and(p_pred < thr, p_tgt >= thr).sum().item())
                arr.append(h_c / max(1.0, h_c + fa_c + m_c))

            tmax_maes.append(float(torch.mean(torch.abs(ens_mean[:, :, 1] - y_phys[:, :, 1])).item()))
            rh_maes.append(float(torch.mean(torch.abs(ens_mean[:, :, 3] - y_phys[:, :, 3])).item()))

            # Bivariate P-RH correlation
            p_f = p_pred.reshape(-1)
            rh_f = ens_mean[:, :, 3].reshape(-1)
            cov = torch.mean((p_f - p_f.mean()) * (rh_f - rh_f.mean()))
            p_rh_corrs.append(float((cov / (p_f.std() * rh_f.std() + 1e-8)).item()))

            # Laplacian energy
            for bi in range(b):
                for di in range(7):
                    e_p = compute_laplacian_energy(p_pred[bi, di])
                    e_t = compute_laplacian_energy(p_tgt[bi, di])
                    lap_ratios.append(e_p / max(1e-6, e_t))

            # 2. Distribution Mode: K=2, S=16, eta=0.0 (32 NFE)
            d_mems = []
            for k in range(2):
                d_mems.append(m.sample(h, f, terr, num_steps=16, eta=0.0, seed=5000 + b_idx * 10 + k))
            d_stack = torch.stack(d_mems, dim=0)

            # Invert K=2
            d_p_list = []
            for k in range(2):
                p_k = torch.clamp(torch.exp(d_stack[k, :, :, 0] * p_s + p_m) - 1.0, min=0.0)
                d_p_list.append(p_k)
            d_p = torch.stack(d_p_list, dim=0)

            # Fair-CRPS (K=2)
            mae_part = torch.mean(torch.abs(d_p - p_tgt.unsqueeze(0)), dim=0)
            ens_diff = torch.abs(d_p[0] - d_p[1])
            crps_k2 = mae_part - (0.25 * ens_diff)
            crps_list.append(float(torch.mean(crps_k2).item()))

            spread_k2 = float(torch.mean(torch.std(d_p, dim=0)).item())
            rmse_k2 = float(torch.sqrt(torch.mean((d_p.mean(dim=0) - p_tgt)**2)).item())
            spread_list.append(spread_k2)
            rmse_list.append(rmse_k2)

    ssr = float(np.mean(spread_list)) / max(1e-6, float(np.mean(rmse_list)))
    return {
        "cubes": len(loader.dataset),
        "wet_mae_mm": round(float(np.mean(wet_maes)), 2),
        "csi15": round(float(np.mean(csi15s)), 4),
        "csi30": round(float(np.mean(csi30s)), 4),
        "csi50": round(float(np.mean(csi50s)), 4),
        "fair_crps_mm": round(float(np.mean(crps_list)), 3),
        "ssr": round(ssr, 3),
        "laplacian_retention": round(float(np.mean(lap_ratios)), 3),
        "tmax_mae": round(float(np.mean(tmax_maes)), 3),
        "rh_mae": round(float(np.mean(rh_maes)), 3),
        "p_rh_corr": round(float(np.mean(p_rh_corrs)), 3),
    }

print("[*] Running Canonical Physical Evaluation on 2022 Validation Split (122 cubes)...")
val_results = evaluate_canonical_physical(model_l, val_loader, ds_val.stats)
print(f"[+] 2022 Validation: Wet-MAE={val_results['wet_mae_mm']} mm | CSI@30={val_results['csi30']} | Fair-CRPS={val_results['fair_crps_mm']} mm | SSR={val_results['ssr']}")


In [ ]:
# Cell 7: Quarantined 2023 El Nino Holdout Generalization Evaluation
ds_test = SpatiotemporalDownscalingDataset(zarr_file, index_file, stats_file, split="test")
test_loader = DataLoader(ds_test, batch_size=2, shuffle=False)
print(f"[*] Unquarantining 2023 El Nino Holdout Set ({len(ds_test)} forecast cubes)...")

test_results = evaluate_canonical_physical(model_l, test_loader, ds_test.stats)
print(f"[+] 2023 Holdout: Wet-MAE={test_results['wet_mae_mm']} mm | CSI@30={test_results['csi30']} | Fair-CRPS={test_results['fair_crps_mm']} mm | SSR={test_results['ssr']}")

# Compute generalization gap
gen_gap_mae = round(test_results["wet_mae_mm"] - val_results["wet_mae_mm"], 2)
gen_gap_csi30 = round(test_results["csi30"] - val_results["csi30"], 4)
gen_gap_crps = round(test_results["fair_crps_mm"] - val_results["fair_crps_mm"], 3)

summary = {
    "model": "Dense-L Champion (52.0M parameters)",
    "val_2022": val_results,
    "test_2023_holdout": test_results,
    "generalization_gap": {
        "wet_mae_delta_mm": gen_gap_mae,
        "csi30_delta": gen_gap_csi30,
        "fair_crps_delta_mm": gen_gap_crps,
    }
}

with open(reports_dir / "sprint10_final_scaling_results.json", "w") as f:
    json.dump(summary, f, indent=2)
with open(working_dir / "sprint10_final_scaling_results.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 60)
print("SPRINT 10 FINAL SUMMARY:")
print(f"  2022 Val Wet-MAE:     {val_results['wet_mae_mm']} mm (Fair-CRPS: {val_results['fair_crps_mm']} mm, CSI@30: {val_results['csi30']})")
print(f"  2023 Holdout Wet-MAE: {test_results['wet_mae_mm']} mm (Fair-CRPS: {test_results['fair_crps_mm']} mm, CSI@30: {test_results['csi30']})")
print(f"  Generalization Delta: {gen_gap_mae:+.2f} mm Wet-MAE ({gen_gap_csi30:+.4f} CSI@30)")
print("=" * 60)
print("[+] SPRINT 10 EXECUTION COMPLETE!")
